# Olympics Data Lake 

Pipeline completo: **raw → bronze → gold**


In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

BASE = '../..'

---
## 1. RAW 

In [ ]:
df_hist = pd.read_csv(f'{BASE}/raw/olympics_historico.csv', low_memory=False)
df_hist['year'] = df_hist['edition'].str.extract(r'(\d{4})').astype(int)
print(f'Histórico: {df_hist.shape}')
df_hist.head(3)

In [ ]:
df_2024 = pd.read_csv(f'{BASE}/raw/olympics_paris2024.csv')
print(f'Paris 2024: {df_2024.shape}')
df_2024.head(3)

---
## 2. BRONZE – Integração

In [ ]:
# JOIN: histórico + Paris 2024
df_hist_med = df_hist[df_hist['medal'].notna()].copy()
df_hist_med = df_hist_med.rename(columns={'country_noc':'country_code','sport':'discipline','athlete':'name'})
df_hist_med['source'] = 'historico'
df_hist_med['edition_year'] = df_hist_med['year']

df_2024n = df_2024.copy()
df_2024n['medal'] = df_2024n['medal_type'].map({'Gold Medal':'Gold','Silver Medal':'Silver','Bronze Medal':'Bronze'})
df_2024n['source'] = 'paris2024'
df_2024n['edition_year'] = 2024
df_2024n['edition'] = '2024 Summer Olympics'

medals = pd.concat([
    df_hist_med[['edition','edition_year','country_code','discipline','event','name','medal','is_team_sport','source']],
    df_2024n[['edition','edition_year','country_code','discipline','event','name','medal','source']].assign(is_team_sport=None)
], ignore_index=True)
print(f'Medalhas consolidadas: {len(medals):,}')
medals.head()

In [ ]:
modalities = df_hist.groupby(['year','edition','sport']).agg(
    total_events=('event','nunique'),
    total_athletes=('athlete_id','nunique'),
    total_medals=('medal', lambda x: x.notna().sum())
).reset_index()
print('Modalidades:', modalities.shape)
modalities.head()

In [ ]:
gender_df = df_2024.groupby(['gender','discipline']).agg(
    total_medalhas=('medal_type','count'),
    total_atletas=('code','nunique'),
    ouro=('medal_type', lambda x: (x=='Gold Medal').sum()),
    prata=('medal_type', lambda x: (x=='Silver Medal').sum()),
    bronze=('medal_type', lambda x: (x=='Bronze Medal').sum())
).reset_index()
gender_df['gender_label'] = gender_df['gender'].map({'M':'Masculino','F':'Feminino'})
print('Gênero/Disciplina:', gender_df.shape)

---
## 3. GOLD – Análises e Visualizações

In [ ]:
# Quadro de medalhas
medal_board = df_2024.groupby(['country_code','country']).apply(
    lambda g: pd.Series({'ouro':(g['medal_type']=='Gold Medal').sum(),
                          'prata':(g['medal_type']=='Silver Medal').sum(),
                          'bronze':(g['medal_type']=='Bronze Medal').sum(),
                          'total':len(g)})
).reset_index()
medal_board = medal_board.sort_values(['ouro','prata','bronze'],ascending=False).reset_index(drop=True)
medal_board['rank'] = medal_board.index + 1
medal_board.head(10)

In [ ]:
# Plot 1: Top 15 países
top15 = medal_board.head(15)
fig, ax = plt.subplots(figsize=(12,7))
x = np.arange(len(top15)); w = 0.25
ax.bar(x-w, top15['ouro'],  w, label='Ouro',   color='#FFD700', edgecolor='gray', lw=0.5)
ax.bar(x,   top15['prata'], w, label='Prata',  color='#C0C0C0', edgecolor='gray', lw=0.5)
ax.bar(x+w, top15['bronze'],w, label='Bronze', color='#CD7F32', edgecolor='gray', lw=0.5)
ax.set_xticks(x); ax.set_xticklabels(top15['country_code'],rotation=45,ha='right')
ax.set_title('Quadro de Medalhas – Paris 2024 (Top 15)',fontsize=14,fontweight='bold')
ax.set_ylabel('Nº de Medalhas'); ax.legend(); ax.grid(axis='y',alpha=0.3)
plt.tight_layout()
plt.savefig(f'{BASE}/gold/analise_medalhas/medalhas_plot.png',dpi=150)
plt.show(); print('salvo')

In [ ]:
# Plot 2: Gênero
gen = df_2024.groupby('gender')['medal_type'].value_counts().unstack(fill_value=0)
gen.index = ['Feminino' if i=='F' else 'Masculino' for i in gen.index]
fig, ax = plt.subplots(figsize=(8,5))
gen[['Gold Medal','Silver Medal','Bronze Medal']].plot(kind='bar',ax=ax,
    color=['#FFD700','#C0C0C0','#CD7F32'],edgecolor='gray',lw=0.5)
ax.set_title('Medalhas por Gênero – Paris 2024',fontsize=13,fontweight='bold')
ax.set_xticklabels(gen.index,rotation=0); ax.legend(['Ouro','Prata','Bronze'])
ax.grid(axis='y',alpha=0.3); plt.tight_layout()
plt.savefig(f'{BASE}/gold/analise_genero/genero_plot.png',dpi=150)
plt.show()

In [ ]:
# Plot 3: Modalidades históricas
top_sports = df_hist.groupby('sport')['athlete_id'].nunique().sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(10,6))
top_sports.plot(kind='barh',ax=ax,color='#4C72B0',edgecolor='gray',lw=0.5)
ax.set_title('Top 10 Modalidades por Atletas – 1896 a 2022',fontsize=13,fontweight='bold')
ax.set_xlabel('Nº de Atletas Únicos'); ax.grid(axis='x',alpha=0.3)
plt.tight_layout()
plt.savefig(f'{BASE}/gold/analise_modalidades/modalidades_plot.png',dpi=150)
plt.show()

---
## 4. Exportação

In [ ]:
medals.to_csv(f'{BASE}/bronze/medalhas_1986_2024.csv',index=False)
modalities.to_csv(f'{BASE}/bronze/modalidades_1986_2024.csv',index=False)
gender_df.to_csv(f'{BASE}/bronze/atletas_por_sexo.csv',index=False)
medal_board.to_csv(f'{BASE}/gold/analise_medalhas/medalhas_summary.csv',index=False)
print('arquivos exportados!')